In [8]:
#import cell
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from prettytable import PrettyTable
from utils import stats, exploration_plots
from preprocessing import preprocessing, preprocessing_temp, preprocessing_temp2
import sys
import importlib
from sklearn.preprocessing import OneHotEncoder
import re

In [9]:
import preprocessing
importlib.reload(preprocessing)

<module 'preprocessing' from 'c:\\Users\\tomma\\OneDrive - Politecnico di Torino\\Desktop\\università\\magistrale\\primo anno\\primo semestre\\dataScienceLab\\project\\preprocessing.py'>

**importing the data**

In [10]:
df = pd.read_csv("development.csv", header = "infer")
evaluate_df = pd.read_csv("evaluation.csv", header = "infer")

#df['id'].to_csv("development_ids.csv", index=False)


In [ ]:
def estimate_from_field(df, field):
    series = df[field]
    max_vals, min_vals, std_vals, mean_vals = [], [], [], []
    for line in series:
        matches = re.findall(r'\$\d+(?:\.\d+)?[^\s]*', str(line))
        values = []
        if matches:
            for value in matches:
                match = re.findall(r'\$(\d+(?:\.\d+)?)', value)
                values.append(float(match[0]) if match else -1)
        max_vals.append(np.max(values) if values else -1)
        min_vals.append(np.min(values) if values else -1)
        std_vals.append(np.std(values) if values else -1)
        mean_vals.append(np.mean(values) if values else -1)
    return max_vals, min_vals, std_vals, mean_vals

df["max_from_body"], df["min_from_body"], df["std_from_body"], df["mean_from_body"] = estimate_from_field(df, "body")
df["max_from_title"], df["min_from_title"], df["std_from_title"], df["mean_from_title"] = estimate_from_field(df, "title")
evaluate_df["max_from_body"], evaluate_df["min_from_body"], evaluate_df["std_from_body"], evaluate_df["mean_from_body"] = estimate_from_field(evaluate_df, "body")
evaluate_df["max_from_title"], evaluate_df["min_from_title"], evaluate_df["std_from_title"], evaluate_df["mean_from_title"] = estimate_from_field(evaluate_df, "title")


    

**Starting the data cleaning**

In [14]:
missing_per_feature_dev = df.isnull().sum()
print(missing_per_feature_dev)

category           0
bathrooms          0
bedrooms           0
fee                0
has_photo          0
pets_allowed       0
square_feet        0
latitude           0
longitude          0
source             0
time               0
id                 0
max_from_body      0
min_from_body      0
std_from_body      0
mean_from_body     0
max_from_title     0
min_from_title     0
std_from_title     0
mean_from_title    0
dtype: int64


In [13]:
df = preprocessing_temp2(df)
y = df["price"]
df.drop(columns = ["price"], inplace = True)
#df = df.fillna(1)

In [69]:
df.shape, y.shape

((79564, 21), (79564,))

In [86]:
# categorical_cols = df.select_dtypes(include=['object']).columns

# Fit the encoder on the training data
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# encoder.fit(df[["category","has_photo", "pets_allowed", "source"]])
# x_encoded = encoder.transform(df[["category","has_photo", "pets_allowed", "source"]])
# # Split back into train and test

# df_one_hot = pd.DataFrame(x_encoded) 
# print(df_one_hot.shape) 
# df_removed_col = df.drop(columns=["category","has_photo", "pets_allowed", "source"])
# df_with_one_hot = pd.concat([df_removed_col, df_one_hot], axis=0)


In [15]:
x_encoded = df.drop(columns=["category","has_photo", "pets_allowed", "source", "fee"])
# x_encoded = df_with_one_hot


In [88]:
# y_encoded = encoder.transform(evaluate_df[["category","has_photo", "pets_allowed", "source"]])
# evaluate_df_one_hot = pd.DataFrame(y_encoded)
# evaluate_df_removed_col = evaluate_df.drop(columns=["category","has_photo", "pets_allowed", "source"])
# evaluate_df_with_one_hot = pd.concat([evaluate_df_removed_col, evaluate_df_one_hot], axis=1)


In [16]:
x_encoded.shape, y.shape

((79425, 15), (79425,))

In [17]:
x_encoded.info()
# print(x_encoded["fee"].unique())  

<class 'pandas.core.frame.DataFrame'>
Index: 79425 entries, 0 to 79588
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        79425 non-null  float64
 1   bedrooms         79425 non-null  float64
 2   square_feet      79425 non-null  int64  
 3   latitude         79425 non-null  float64
 4   longitude        79425 non-null  float64
 5   time             79425 non-null  int64  
 6   id               79425 non-null  int64  
 7   max_from_body    79425 non-null  float64
 8   min_from_body    79425 non-null  float64
 9   std_from_body    79425 non-null  float64
 10  mean_from_body   79425 non-null  float64
 11  max_from_title   79425 non-null  float64
 12  min_from_title   79425 non-null  float64
 13  std_from_title   79425 non-null  float64
 14  mean_from_title  79425 non-null  float64
dtypes: float64(12), int64(3)
memory usage: 9.7 MB


In [18]:

# y = x_encoded["price"]
# combined_encoded = x_encoded.drop(columns="price")

X_train, X_test, y_train, y_test = train_test_split(x_encoded, y, test_size= 0.2 ,shuffle=True, random_state=42)



In [19]:
# Separate features and target
table = PrettyTable()

rfr = RandomForestRegressor(n_jobs=-1)
rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
table.field_names = ["Model", "MAE"]
table.add_row(["Random Forest", mae])
print(table)


+---------------+--------------------+
|     Model     |        MAE         |
+---------------+--------------------+
| Random Forest | 160.43426883852692 |
+---------------+--------------------+


In [20]:
# # Is it radom?!
# y_pred2 = np.random.permutation(y_pred)
# mae = mean_absolute_error(y_test, y_pred2)
# table.field_names = ["Model", "MAE"]
# table.add_row(["Random Forest", mae])
# print(table)


In [ ]:
# #evaluate_df = preprocessing_temp2(evaluate_df)
# display(evaluate_df)
# evaluate_df = evaluate_df.drop(columns=["bedrooms","bathrooms"])



In [21]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# Concatenate train and test to ensure consistent encoding
eval_encoded = pd.get_dummies(evaluate_df, columns=categorical_cols)

In [22]:
importances = rfr.feature_importances_
feature_names = X_train.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(feat_imp.head(20))  # Show top 20 most important features

longitude          0.335629
square_feet        0.263344
latitude           0.145211
bathrooms          0.060998
time               0.054390
id                 0.035746
min_from_body      0.029077
mean_from_body     0.023709
max_from_body      0.015349
bedrooms           0.012487
std_from_body      0.011389
mean_from_title    0.004211
max_from_title     0.004206
min_from_title     0.003855
std_from_title     0.000396
dtype: float64


In [23]:
for col in evaluate_df.columns:
    if col not in x_encoded.columns:
        evaluate_df.drop(columns=col, inplace=True) 

In [24]:
evaluate_df.columns == x_encoded.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True])

In [25]:
rfr_online = RandomForestRegressor(n_jobs=-1)
rfr_online.fit(x_encoded,y)

# Now you can safely predict
y_pred = rfr_online.predict(evaluate_df)
results_df = pd.DataFrame({
    "Id": evaluate_df["id"],
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)


In [ ]:
#api_key: c97d8bd8e2dcfa84c1e1c72140d1582f23e136d1ba0fd19d8a39915e7db6207e

In [ ]:
importances = rfr_online.feature_importances_
feature_names = X_train.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(feat_imp.head(20))  # Show top 20 most important features

da fare
- assegnazioni valori a bathrooms and bedrooms
- distribuzione stati nel dataset (evaluation e train)
- gestire i campi title e body con tf-Idf
